# Exploration under Sparse Rewards: $\varepsilon$-greedy vs Intrinsic Motivation (RND / ICM)

*A simple RL teaching notebook --- Colab-runnable, CPU-only, no `gym`.*

---

## The exploration problem

Most "easy" RL successes lean on **dense reward**: the environment hands you a
useful learning signal at almost every step, so even crude exploration (act
randomly now and then) eventually stumbles into good behaviour and the value
estimates do the rest.

**Sparse reward** breaks this. Imagine a reward of $+1$ delivered *only* when the
agent reaches a single distant goal cell, and $0$ everywhere else. Until the
agent reaches that cell *at least once*, every transition it sees has reward
$0$. There is **no gradient of improvement** to follow --- the value function is
flat, the policy has nothing to prefer, and the only way to discover the goal is
to literally walk into it by chance.

With $\varepsilon$-greedy exploration, the agent takes a random action with
probability $\varepsilon$ and its (currently arbitrary) greedy action otherwise.
On a flat value landscape this is essentially an **undirected random walk**. The
expected time for a random walk to first reach a far corner of an $N\times N$ grid
grows quickly, and when episodes are **truncated** at a modest step cap the agent
may *never* reach the goal within the training budget. No reward signal
$\Rightarrow$ no learning $\Rightarrow$ failure.

## Intrinsic motivation (curiosity)

The fix that scales is to give the agent its **own** reward for visiting novel
states --- an *intrinsic* bonus added on top of the (sparse) *extrinsic* reward:

$$ r_t^{\text{total}} = r_t^{\text{extrinsic}} + \beta \, r_t^{\text{intrinsic}}. $$

The intrinsic term should be **large for unfamiliar states and decay as a state
becomes familiar**, turning "go somewhere new" into a dense, self-generated
learning signal that systematically pushes the agent outward until it bumps into
the real goal. We compare three signals:

* **Count-based (pseudo-counts).** The cleanest possible novelty:
  $r^{\text{int}}(s') = 1/\sqrt{N(s')}$ where $N(s')$ is how often $s'$ has been
  visited. New states give a big bonus that shrinks monotonically. This is the
  classic exploration-bonus idea (Strehl & Littman; later generalised to
  *pseudo-counts* by Bellemare et al. for high-dim states). We use it as the
  transparent reference that reliably solves the task.

* **RND (Random Network Distillation).** The **neural** way to estimate novelty
  without explicit counts. A *fixed, randomly initialised* target network
  $\bar f(s)$ defines an arbitrary embedding; a *trained* predictor $f_\theta(s)$
  is regressed toward it. The intrinsic reward is the prediction error
  $\lVert f_\theta(s')-\bar f(s')\rVert^2$: high for rarely-seen states
  (predictor hasn't fit them), shrinking wherever the agent has spent time ---
  i.e. a learned soft count. We use the two-value-head trick from the original
  paper (a separate non-episodic head for the intrinsic stream).

* **ICM (Intrinsic Curiosity Module)** *(optional)*. Learns its own feature space
  $\phi(s)$ (via an *inverse* model, so it keeps only action-relevant detail),
  plus a *forward* model predicting $\phi(s')$ from $\phi(s),a$. Intrinsic
  reward = forward-model error $\lVert\hat\phi(s')-\phi(s')\rVert^2$.

## What we'll show

On a sparse four-rooms gridworld, all on CPU in ~1-2 minutes, with everything
seeded:

1. **$\varepsilon$-greedy tabular Q-learning** (pure extrinsic) almost never finds
   the goal within budget --- very low success rate, very late first goal.
2. **Count-based curiosity** drives systematic outward exploration and reliably
   reaches *and then exploits* the goal.
3. **RND** (neural curiosity) finds the goal *earlier* than $\varepsilon$-greedy
   and beats it, illustrating the same mechanism with a learned novelty estimate
   (and exposing why neural curiosity is finicky).

We measure **success rate / episodes-to-first-goal** and draw
**state-visitation heatmaps** (showing $\varepsilon$-greedy huddled near the
start while the curiosity agents spread across the map and reach the goal), plus
the **early intrinsic-reward heatmap** that reveals the exploration frontier.

> Conceptual home: this builds on value-based control from
> [DQN & Rainbow](https://app.notion.com/p/37f95008d76681f396e7ffa88cb535e7)
> and the algorithm taxonomy in
> [Part 2: Kinds of RL Algorithms](https://app.notion.com/p/37895008d76681a8861fc7a364b26b68).


## 1. Setup: imports, config, seeds

We keep the configuration **small** so the whole notebook runs in ~1-2 minutes on
a CPU. `torch` powers the RND / ICM neural nets; the tabular Q-learning and
count-based agents are pure `numpy`. Everything is seeded for reproducibility,
and we print elapsed time at each training stage.


In [ ]:
import time, math, random
from dataclasses import dataclass, field
from typing import List, Tuple, Optional

import numpy as np
import matplotlib.pyplot as plt

# torch is used only for the neural intrinsic-motivation modules (RND/ICM).
import torch
import torch.nn as nn
import torch.nn.functional as F

print("numpy", np.__version__, "| torch", torch.__version__)
torch.set_num_threads(2)  # keep CPU usage modest on Colab


def set_all_seeds(seed: int):
    """Seed python, numpy and torch so runs are reproducible."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


@dataclass
class Config:
    # --- environment ---------------------------------------------------
    grid_size: int = 11          # NxN grid (four-rooms carved into it)
    max_steps: int = 80          # episode truncation (step cap); TIGHT on purpose
    goal_reward: float = 1.0     # +1 only at the goal, 0 elsewhere (sparse!)
    step_penalty: float = 0.0    # keep 0 -> reward is purely sparse
    four_rooms: bool = True      # carve interior walls into 4 rooms w/ doorways

    # --- training ------------------------------------------------------
    episodes: int = 800          # episodes per agent (small but enough to show effect)
    gamma: float = 0.99          # discount (extrinsic value)
    lr_q: float = 0.5            # tabular Q-learning rate (high: deterministic env)
    # epsilon schedule (linear decay) for the Q-learning policy
    eps_start: float = 1.0
    eps_end: float = 0.05        # floor for the pure eps-greedy / count agents
    eps_decay_frac: float = 0.5  # fraction of episodes over which eps decays

    # --- intrinsic motivation -----------------------------------------
    beta_count: float = 0.1      # weight on count-based 1/sqrt(N) bonus
    rnd_scale: float = 20.0      # weight on RND raw prediction-error bonus
    icm_scale: float = 20.0      # weight on ICM forward-error bonus
    gamma_int: float = 0.9       # discount for the NON-episodic intrinsic value head
    eps_floor_int: float = 0.1   # exploration floor for curiosity agents (keep stirring)
    lr_net: float = 3e-3         # predictor / ICM optimiser lr
    feat_dim: int = 32           # embedding dim for RND target/predictor & ICM
    icm_beta_inv: float = 0.2    # ICM: weight of inverse vs forward loss

    # --- experiment ----------------------------------------------------
    seeds: List[int] = field(default_factory=lambda: [0, 1, 2])


CFG = Config()
print(CFG)


## 2. A sparse-reward four-rooms gridworld (numpy, no gym)

A classic hard-exploration toy. The agent lives on an $N\times N$ grid. With
`four_rooms=True` we carve two interior walls (a `+` of walls) into **four rooms
connected by single-cell doorways**, so the agent must thread narrow gaps to move
between rooms --- this makes naive random exploration much slower.

* **State** = the agent's `(row, col)`; we expose it three ways: an integer index
  (for the tabular agents), the raw `(r,c)` coordinate, and a **one-hot vector**
  over cells that the neural modules consume (one-hot makes each cell distinct, so
  RND's predictor must actually *memorise* visited states --- novelty = unfit
  cells).
* **Actions** = `{0:up, 1:down, 2:left, 3:right}`. Moving into a wall or off the
  grid is a no-op (you stay put).
* **Reward** = `goal_reward` **only** on the step that lands on the goal, else
  `step_penalty` (which we set to 0). This is the sparse signal.
* **Start** = top-left corner; **goal** = bottom-right corner (opposite corners,
  two doorways away).
* **Termination**: reaching the goal (`done=True`) or hitting `max_steps`
  (truncated; `done=False`, `truncated=True`). The **tight step cap** is what
  makes $\varepsilon$-greedy fail: an undirected walk rarely threads both doorways
  to the far corner in time.


In [ ]:
class GridWorld:
    """Sparse-reward four-rooms gridworld, implemented directly in numpy.

    Coordinates are (row, col), row 0 at top. `self.walls[r, c] == True` means
    the cell is impassable.
    """
    ACTIONS = np.array([(-1, 0), (1, 0), (0, -1), (0, 1)])  # up, down, left, right
    ACTION_NAMES = ["up", "down", "left", "right"]

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.n = cfg.grid_size
        self.walls = self._build_walls()
        self.start = (0, 0)
        self.goal = (self.n - 1, self.n - 1)
        # ensure start/goal are open
        self.walls[self.start] = False
        self.walls[self.goal] = False
        self.n_actions = 4
        self.n_states = self.n * self.n
        self.reset()

    # --- map construction ---------------------------------------------
    def _build_walls(self):
        n = self.n
        walls = np.zeros((n, n), dtype=bool)
        if not self.cfg.four_rooms:
            return walls
        # Carve a '+' of walls splitting the grid into 4 rooms, one doorway each.
        mid = n // 2
        walls[mid, :] = True   # horizontal divider
        walls[:, mid] = True   # vertical divider
        q = max(1, n // 4)
        doorways = [
            (mid, q),            # left arm of horizontal wall
            (mid, n - 1 - q),    # right arm of horizontal wall
            (q, mid),            # top arm of vertical wall
            (n - 1 - q, mid),    # bottom arm of vertical wall
        ]
        for (r, c) in doorways:
            walls[r, c] = False
        return walls

    # --- state encodings ----------------------------------------------
    def to_index(self, pos: Tuple[int, int]) -> int:
        return pos[0] * self.n + pos[1]

    def from_index(self, idx: int) -> Tuple[int, int]:
        return (idx // self.n, idx % self.n)

    def onehot(self, pos: Tuple[int, int]) -> np.ndarray:
        """One-hot encoding over cells (input for the neural modules)."""
        v = np.zeros(self.n_states, dtype=np.float32)
        v[self.to_index(pos)] = 1.0
        return v

    @property
    def feature_size(self) -> int:
        return self.n_states

    # --- dynamics ------------------------------------------------------
    def reset(self) -> Tuple[int, int]:
        self.pos = self.start
        self.t = 0
        return self.pos

    def step(self, action: int):
        """Apply action. Returns (next_pos, reward, done, truncated)."""
        self.t += 1
        dr, dc = self.ACTIONS[action]
        nr, nc = self.pos[0] + dr, self.pos[1] + dc
        if 0 <= nr < self.n and 0 <= nc < self.n and not self.walls[nr, nc]:
            self.pos = (nr, nc)          # else: stay put (wall / off-grid)
        reward = self.cfg.step_penalty
        done = False
        if self.pos == self.goal:
            reward = self.cfg.goal_reward    # sparse: +1 only here
            done = True
        truncated = (self.t >= self.cfg.max_steps) and not done
        return self.pos, reward, done, truncated


### 2.1 Sanity-check the environment

A couple of asserts to confirm the transition rules and sparse-goal logic before
we train anything.


In [ ]:
_env = GridWorld(CFG)
print(f"grid {_env.n}x{_env.n}, start={_env.start}, goal={_env.goal}, "
      f"actions={_env.n_actions}, onehot_dim={_env.feature_size}")

# (a) reset returns the start; index<->pos round-trips.
assert _env.reset() == _env.start
assert _env.from_index(_env.to_index((3, 4))) == (3, 4)

# (b) moving up/left from the top-left corner is a no-op (off-grid), 0 reward.
_env.reset()
pos, r, d, tr = _env.step(0)  # up
assert pos == (0, 0) and r == 0.0 and not d, "off-grid move => no-op, 0 reward"
pos, r, d, tr = _env.step(2)  # left
assert pos == (0, 0), "off-grid move => no-op"

# (c) reward is sparse: +1 only when stepping ONTO the goal.
_env.reset()
_env.pos = (_env.n - 1, _env.n - 2)  # one cell left of the goal
pos, r, d, tr = _env.step(3)         # step right -> onto goal
assert pos == _env.goal and r == CFG.goal_reward and d, "onto goal => +1 and done"

# (d) episode truncates at the step cap when goal not reached.
_env.reset()
trunc_seen = False
for _ in range(CFG.max_steps + 5):
    _, _, d, tr = _env.step(0)  # keep bumping into the top wall
    if tr:
        trunc_seen = True
        break
assert trunc_seen, "episode must truncate at max_steps"

# (e) one-hot encoding has unit mass at the right index.
oh = _env.onehot((2, 3))
assert oh.shape == (_env.n_states,) and oh.sum() == 1.0 and oh[_env.to_index((2, 3))] == 1.0
print("env sanity checks passed.")


### 2.2 Visualise the map

Black = walls, white = free cells, green = start (S), red = goal (G). Note the
four rooms and the single-cell doorways the agent must thread through, and that
the goal sits two doorways away from the start.


In [ ]:
def plot_map(env, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4))
    img = np.ones((env.n, env.n))
    img[env.walls] = 0.0  # walls black
    ax.imshow(img, cmap="gray", vmin=0, vmax=1)
    ax.scatter([env.start[1]], [env.start[0]], c="lime", s=160, marker="s",
               edgecolors="black", zorder=3)
    ax.scatter([env.goal[1]], [env.goal[0]], c="red", s=200, marker="*",
               edgecolors="black", zorder=3)
    ax.text(env.start[1], env.start[0], "S", ha="center", va="center",
            fontsize=9, fontweight="bold")
    ax.text(env.goal[1], env.goal[0], "G", ha="center", va="center",
            fontsize=9, fontweight="bold", color="white")
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_title(f"Four-rooms gridworld ({env.n}x{env.n})")
    return ax

plot_map(GridWorld(CFG))
plt.tight_layout(); plt.show()


## 3. Training harness (shared by all agents)

Every agent implements the same tiny interface so we can run them through one
loop and compare apples to apples:

* `act(pos)` --- choose an action given the current position.
* `observe(pos, a, r_ext, next_pos, done)` --- learn from one transition. The
  agent internally computes any intrinsic bonus and updates its value estimates /
  networks, and **returns the raw intrinsic reward** (or `None`) for diagnostics.
* `end_episode(ep)` --- per-episode bookkeeping (e.g. epsilon decay).

The harness records, **per episode**, whether the goal was reached (success), and
accumulates a **state-visitation count grid** plus an **early intrinsic-reward
grid**. From these we derive the success-rate curve and episodes-to-first-goal,
averaged over seeds.


In [ ]:
def run_agent(make_agent, cfg: Config, seeds=None, episodes=None, verbose=True):
    """Train `make_agent(env, cfg)` over several seeds; return aggregated metrics.

    Returns a dict with:
      success_curve : (episodes,) mean success indicator over seeds
      first_goal    : list[int] episode index of first goal per seed (-1 if never)
      visitation    : (n,n) summed state-visitation counts over all seeds
      intrinsic_grid: (n,n) mean RAW intrinsic reward per cell, EARLY in training
                      (zeros for the pure eps-greedy agent)
      elapsed       : wall-clock seconds
    """
    seeds = cfg.seeds if seeds is None else seeds
    episodes = cfg.episodes if episodes is None else episodes
    t0 = time.time()

    n = cfg.grid_size
    succ = np.zeros((len(seeds), episodes), dtype=np.float32)
    first_goal = []
    visitation = np.zeros((n, n), dtype=np.float64)
    intr_sum = np.zeros((n, n), dtype=np.float64)
    intr_cnt = np.zeros((n, n), dtype=np.float64)
    early_cut = max(1, episodes // 5)  # "early training" window for intrinsic map

    for si, seed in enumerate(seeds):
        set_all_seeds(seed)
        env = GridWorld(cfg)
        agent = make_agent(env, cfg)
        fg = -1
        for ep in range(episodes):
            pos = env.reset()
            done = trunc = False
            reached = False
            while not (done or trunc):
                a = agent.act(pos)
                npos, r_ext, done, trunc = env.step(a)
                r_int = agent.observe(pos, a, r_ext, npos, done)
                visitation[pos] += 1.0
                if ep < early_cut and r_int is not None:
                    intr_sum[pos] += float(r_int)
                    intr_cnt[pos] += 1.0
                pos = npos
                if done:
                    reached = True
            agent.end_episode(ep)
            succ[si, ep] = 1.0 if reached else 0.0
            if reached and fg < 0:
                fg = ep
        first_goal.append(fg)
        if verbose:
            sr = succ[si].mean()
            fgs = fg if fg >= 0 else "never"
            print(f"  seed {seed}: success-rate={sr:.3f}, first goal @ ep {fgs}")

    intrinsic_grid = np.divide(intr_sum, np.maximum(intr_cnt, 1.0))
    return {
        "success_curve": succ.mean(axis=0),
        "success_per_seed": succ,
        "first_goal": first_goal,
        "visitation": visitation,
        "intrinsic_grid": intrinsic_grid,
        "elapsed": time.time() - t0,
    }


def moving_average(x, w=50):
    if len(x) < w:
        return x
    return np.convolve(x, np.ones(w) / w, mode="valid")


## 4. Agent 1 --- $\varepsilon$-greedy tabular Q-learning (pure extrinsic)

Vanilla tabular Q-learning. The Q-table is `Q[state_index, action]`, updated by
the standard TD rule

$$ Q(s,a) \leftarrow Q(s,a) + \alpha\big[r + \gamma \max_{a'} Q(s',a') - Q(s,a)\big]. $$

Exploration is **$\varepsilon$-greedy** with linearly-decaying $\varepsilon$. The
reward $r$ is the **raw extrinsic** reward only --- so until a random walk happens
to reach the goal, every update uses $r=0$ on a flat table and *nothing changes*.
This is exactly the failure mode we want to expose.


In [ ]:
class QLearningAgent:
    """Tabular Q-learning with epsilon-greedy exploration (extrinsic reward only)."""
    def __init__(self, env: GridWorld, cfg: Config):
        self.env = env
        self.cfg = cfg
        self.Q = np.zeros((env.n_states, env.n_actions), dtype=np.float64)
        self.eps = cfg.eps_start
        self._decay = (cfg.eps_start - cfg.eps_end) / max(
            1, int(cfg.eps_decay_frac * cfg.episodes))

    def act(self, pos):
        if np.random.rand() < self.eps:
            return np.random.randint(self.env.n_actions)  # explore
        q = self.Q[self.env.to_index(pos)]
        # random tie-break so an all-zero row doesn't always pick action 0
        return int(np.random.choice(np.flatnonzero(q == q.max())))

    def observe(self, pos, a, r_ext, npos, done):
        s = self.env.to_index(pos)
        ns = self.env.to_index(npos)
        target = r_ext + (0.0 if done else self.cfg.gamma * self.Q[ns].max())
        self.Q[s, a] += self.cfg.lr_q * (target - self.Q[s, a])
        return None  # no intrinsic reward

    def end_episode(self, ep):
        self.eps = max(self.cfg.eps_end, self.eps - self._decay)


## 5. Agent 2 --- Count-based curiosity (the clean reference)

The simplest possible intrinsic motivation: keep an explicit visit count $N(s)$
and reward novelty with

$$ r^{\text{int}}(s') = \frac{1}{\sqrt{N(s')}}. $$

A brand-new state gives a bonus of $1$; the bonus shrinks **monotonically and
predictably** as the state is revisited. We add it to the extrinsic reward and
learn one tabular Q-table on $r = r^{\text{ext}} + \beta\, r^{\text{int}}$. This
makes the value landscape *non-flat* immediately: unexplored regions look
valuable, so the greedy policy walks toward them, systematically expanding the
frontier until it threads the doorways and hits the real goal --- after which the
$+1$ dominates and the policy exploits. Transparent, reliable, and the yardstick
against which we read the neural method.

> In high-dimensional state spaces you can't tabulate $N(s)$; **pseudo-counts**
> (Bellemare et al. 2016) and **RND** (next section) are two ways to *estimate*
> this novelty with a neural network.


In [ ]:
class CountAgent:
    """Tabular Q-learning + count-based 1/sqrt(N) intrinsic reward."""
    def __init__(self, env: GridWorld, cfg: Config):
        self.env = env
        self.cfg = cfg
        self.Q = np.zeros((env.n_states, env.n_actions), dtype=np.float64)
        self.N = np.zeros(env.n_states, dtype=np.float64)  # visit counts
        self.eps = cfg.eps_start
        self._decay = (cfg.eps_start - cfg.eps_end) / max(
            1, int(cfg.eps_decay_frac * cfg.episodes))

    def act(self, pos):
        if np.random.rand() < self.eps:
            return np.random.randint(self.env.n_actions)
        q = self.Q[self.env.to_index(pos)]
        return int(np.random.choice(np.flatnonzero(q == q.max())))

    def observe(self, pos, a, r_ext, npos, done):
        ns = self.env.to_index(npos)
        self.N[ns] += 1.0
        r_int = 1.0 / math.sqrt(self.N[ns])            # novelty bonus
        r = r_ext + self.cfg.beta_count * r_int        # extrinsic + beta * intrinsic
        s = self.env.to_index(pos)
        target = r + (0.0 if done else self.cfg.gamma * self.Q[ns].max())
        self.Q[s, a] += self.cfg.lr_q * (target - self.Q[s, a])
        return r_int

    def end_episode(self, ep):
        self.eps = max(self.cfg.eps_end, self.eps - self._decay)


## 6. Agent 3 --- RND-augmented Q-learning (neural curiosity)

**Random Network Distillation** estimates novelty *without* explicit counts:

1. **Target net** $\bar f$: a small MLP with **frozen random weights**, mapping a
   state's one-hot to a random $d$-dim embedding. Never trained.
2. **Predictor net** $f_\theta$: same architecture, **trained** to match $\bar f$
   on visited states via MSE.
3. **Intrinsic reward**: $r^{\text{int}} = \lVert f_\theta(s')-\bar f(s')\rVert^2$.
   States the predictor hasn't fit yet (novel) give large error; revisiting a
   region lets the predictor catch up and the bonus **decays toward 0** --- a
   *learned soft count*.

Two important design choices from the original paper that we keep:

* **Two value heads.** We learn a **separate** intrinsic value $Q_i$ with its own
  (lower, **non-episodic**) discount $\gamma_{\text{int}}$, and act greedily on
  $Q_e + Q_i$. Folding a *decaying* bonus into the single episodic Q-target tends
  to leave stale high values in already-explored interior cells, trapping the
  agent; a dedicated non-episodic intrinsic head largely avoids this.
* **A small exploration floor** ($\varepsilon$ never decays to 0) keeps the agent
  stirring so the predictor keeps seeing the frontier.

All nets are tiny and run fine on CPU. (Compared to the tabular count above, RND
is a *noisier* novelty estimate --- which is exactly why it's instructive.)


In [ ]:
class MLP(nn.Module):
    """Tiny 2-hidden-layer MLP used for RND target/predictor and ICM heads."""
    def __init__(self, in_dim, out_dim, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        return self.net(x)


class RNDAgent:
    """Q-learning + Random Network Distillation intrinsic reward (two value heads)."""
    def __init__(self, env: GridWorld, cfg: Config):
        self.env = env
        self.cfg = cfg
        # separate extrinsic (episodic) and intrinsic (non-episodic) value tables
        self.Qe = np.zeros((env.n_states, env.n_actions), dtype=np.float64)
        self.Qi = np.zeros((env.n_states, env.n_actions), dtype=np.float64)
        self.eps = cfg.eps_start
        self._decay = (cfg.eps_start - cfg.eps_floor_int) / max(
            1, int(cfg.eps_decay_frac * cfg.episodes))

        d = env.feature_size
        self.target = MLP(d, cfg.feat_dim)      # frozen random net
        self.predictor = MLP(d, cfg.feat_dim)   # trained to match it
        for p in self.target.parameters():
            p.requires_grad_(False)
        self.opt = torch.optim.Adam(self.predictor.parameters(), lr=cfg.lr_net)

    def _feat(self, pos):
        return torch.from_numpy(self.env.onehot(pos)).float().unsqueeze(0)

    def act(self, pos):
        if np.random.rand() < self.eps:
            return np.random.randint(self.env.n_actions)
        s = self.env.to_index(pos)
        q = self.Qe[s] + self.Qi[s]             # combine both value heads
        return int(np.random.choice(np.flatnonzero(q == q.max())))

    def _intrinsic(self, npos):
        """Prediction error at next state; also takes a predictor grad step."""
        x = self._feat(npos)
        with torch.no_grad():
            tgt = self.target(x)
        pred = self.predictor(x)
        err = (pred - tgt).pow(2).mean()        # scalar MSE for this state
        raw = float(err.detach())               # read BEFORE updating predictor
        self.opt.zero_grad(); err.backward(); self.opt.step()
        return raw

    def observe(self, pos, a, r_ext, npos, done):
        raw = self._intrinsic(npos)
        s = self.env.to_index(pos)
        ns = self.env.to_index(npos)
        # extrinsic head: standard episodic bootstrap (terminal at the goal)
        te = r_ext + (0.0 if done else self.cfg.gamma * self.Qe[ns].max())
        self.Qe[s, a] += self.cfg.lr_q * (te - self.Qe[s, a])
        # intrinsic head: novelty reward, LOW discount, NEVER terminal
        ti = self.cfg.rnd_scale * raw + self.cfg.gamma_int * self.Qi[ns].max()
        self.Qi[s, a] += self.cfg.lr_q * (ti - self.Qi[s, a])
        return raw  # raw intrinsic for the heatmap / diagnostics

    def end_episode(self, ep):
        self.eps = max(self.cfg.eps_floor_int, self.eps - self._decay)


## 7. Agent 4 *(optional)* --- ICM forward-model curiosity

The **Intrinsic Curiosity Module** learns its *own* feature space instead of
using a random one:

* **Encoder** $\phi(s)$ maps the one-hot state to an embedding.
* **Inverse model** predicts the action $a_t$ from $\big(\phi(s_t),\phi(s_{t+1})\big)$.
  Training the encoder through this head forces $\phi$ to keep only the parts of
  the state the agent can *control* / that matter for action --- discarding
  uncontrollable noise (the "noisy-TV" robustness argument).
* **Forward model** predicts $\hat\phi(s_{t+1})$ from $\big(\phi(s_t), a_t\big)$.
* **Intrinsic reward** = forward prediction error
  $\lVert\hat\phi(s_{t+1})-\phi(s_{t+1})\rVert^2$: high where the learned dynamics
  are still surprising (novel transitions).

We reuse the **same two-value-head Q-core** as RND; only the novelty signal
changes. Loss = `icm_beta_inv * inverse_loss + (1 - icm_beta_inv) * forward_loss`.


In [ ]:
class ICMAgent:
    """Q-learning + Intrinsic Curiosity Module (forward-model surprise, two heads)."""
    def __init__(self, env: GridWorld, cfg: Config):
        self.env = env
        self.cfg = cfg
        self.Qe = np.zeros((env.n_states, env.n_actions), dtype=np.float64)
        self.Qi = np.zeros((env.n_states, env.n_actions), dtype=np.float64)
        self.eps = cfg.eps_start
        self._decay = (cfg.eps_start - cfg.eps_floor_int) / max(
            1, int(cfg.eps_decay_frac * cfg.episodes))

        d = env.feature_size
        fd = cfg.feat_dim
        self.na = env.n_actions
        self.encoder = MLP(d, fd)                  # phi(s)
        self.inverse = MLP(2 * fd, self.na)        # (phi_s, phi_s') -> action logits
        self.forward_m = MLP(fd + self.na, fd)     # (phi_s, a_onehot) -> phi_s'_hat
        params = (list(self.encoder.parameters())
                  + list(self.inverse.parameters())
                  + list(self.forward_m.parameters()))
        self.opt = torch.optim.Adam(params, lr=cfg.lr_net)

    def _feat(self, pos):
        return torch.from_numpy(self.env.onehot(pos)).float().unsqueeze(0)

    def _onehot_a(self, a):
        v = torch.zeros(1, self.na); v[0, a] = 1.0; return v

    def act(self, pos):
        if np.random.rand() < self.eps:
            return np.random.randint(self.env.n_actions)
        s = self.env.to_index(pos)
        q = self.Qe[s] + self.Qi[s]
        return int(np.random.choice(np.flatnonzero(q == q.max())))

    def _intrinsic(self, pos, a, npos):
        s = self._feat(pos); ns = self._feat(npos); a_oh = self._onehot_a(a)
        phi_s = self.encoder(s)
        phi_ns = self.encoder(ns)
        pred_ns = self.forward_m(torch.cat([phi_s, a_oh], dim=1))   # forward model
        fwd_err = (pred_ns - phi_ns.detach()).pow(2).mean()
        logits = self.inverse(torch.cat([phi_s, phi_ns], dim=1))     # inverse model
        inv_loss = F.cross_entropy(logits, torch.tensor([a]))
        loss = (self.cfg.icm_beta_inv * inv_loss
                + (1 - self.cfg.icm_beta_inv) * fwd_err)
        raw = float(fwd_err.detach())             # read BEFORE update
        self.opt.zero_grad(); loss.backward(); self.opt.step()
        return raw

    def observe(self, pos, a, r_ext, npos, done):
        raw = self._intrinsic(pos, a, npos)
        s = self.env.to_index(pos)
        ns = self.env.to_index(npos)
        te = r_ext + (0.0 if done else self.cfg.gamma * self.Qe[ns].max())
        self.Qe[s, a] += self.cfg.lr_q * (te - self.Qe[s, a])
        ti = self.cfg.icm_scale * raw + self.cfg.gamma_int * self.Qi[ns].max()
        self.Qi[s, a] += self.cfg.lr_q * (ti - self.Qi[s, a])
        return raw

    def end_episode(self, ep):
        self.eps = max(self.cfg.eps_floor_int, self.eps - self._decay)


## 8. Train all agents

Same env, same budget --- the **only** difference is the exploration / reward
signal. We run each over all seeds and time it. (RND/ICM do a tiny neural step per
transition, so they take a bit longer than the tabular agents, but the whole
thing still finishes in ~1-2 minutes on CPU.)


In [ ]:
results = {}

print("Training epsilon-greedy Q-learning (pure extrinsic)...")
results["eps-greedy"] = run_agent(lambda e, c: QLearningAgent(e, c), CFG)
print(f"  elapsed: {results['eps-greedy']['elapsed']:.1f}s\n")

print("Training count-based curiosity (1/sqrt(N))...")
results["count"] = run_agent(lambda e, c: CountAgent(e, c), CFG)
print(f"  elapsed: {results['count']['elapsed']:.1f}s\n")

print("Training RND-augmented Q-learning (neural curiosity)...")
results["RND"] = run_agent(lambda e, c: RNDAgent(e, c), CFG)
print(f"  elapsed: {results['RND']['elapsed']:.1f}s\n")

# Optional 4th agent (ICM). Uncomment to include it (adds ~40s). ICM is the most
# finicky here: it learns its OWN feature space online from one sample at a time,
# so on this tiny one-hot toy its novelty signal is the noisiest of the three and
# it solves the task only on some seeds -- a fair illustration of why forward-model
# curiosity needs more care than RND or tabular counts.
# print("Training ICM-augmented Q-learning (optional)...")
# results["ICM"] = run_agent(lambda e, c: ICMAgent(e, c), CFG)
# print(f"  elapsed: {results['ICM']['elapsed']:.1f}s\n")

total = sum(r["elapsed"] for r in results.values())
print(f"TOTAL training time: {total:.1f}s")


## 9. Headline metrics: success rate & episodes-to-first-goal

The two numbers that tell the story:

* **Overall success rate** --- fraction of episodes that reached the goal.
* **Episodes-to-first-goal** --- how long until the agent *first* sees any reward.
  $\varepsilon$-greedy should be very late (or "never"); the curiosity agents find
  it much sooner, and count-based then converges to a near-perfect policy.


In [ ]:
print(f"{'method':<12}{'success-rate':>14}{'episodes-to-first-goal (per seed)':>40}")
print("-" * 66)
for name, res in results.items():
    sr = res["success_curve"].mean()
    fg = ["never" if x < 0 else str(x) for x in res["first_goal"]]
    print(f"{name:<12}{sr:>14.3f}{', '.join(fg):>40}")


## 10. Plot (a): success rate vs training episodes

Smoothed (moving-average) per-episode success indicator, averaged over seeds.
Watch $\varepsilon$-greedy stay flat near the floor while the curiosity agents
climb once their *directed* exploration finds the goal --- count-based cleanly,
RND/ICM more raggedly (noisier novelty estimates).


In [ ]:
plt.figure(figsize=(7, 4.4))
colors = {"eps-greedy": "tab:red", "count": "tab:purple",
          "RND": "tab:blue", "ICM": "tab:green"}
for name, res in results.items():
    curve = res["success_curve"]
    ma = moving_average(curve, w=max(10, len(curve) // 30))
    plt.plot(np.arange(len(ma)), ma, label=name, color=colors.get(name), lw=2)
plt.xlabel("episode"); plt.ylabel("success rate (moving avg)")
plt.title("Reaching the sparse goal: epsilon-greedy vs curiosity")
plt.ylim(-0.02, 1.02); plt.legend(); plt.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 11. Plot (b): state-visitation heatmaps

Where did each agent actually *go*? We sum visitation counts over all seeds and
training (log-scaled for readability). The expected picture:

* **$\varepsilon$-greedy**: a bright blob hugging the **start** room --- it
  diffuses slowly and rarely escapes through the doorways to the far corner.
* **count / RND / ICM**: visitation **spread across the rooms**, with mass
  reaching the **goal** corner --- systematic, curiosity-driven coverage.


In [ ]:
env = GridWorld(CFG)
ncol = len(results)
fig, axes = plt.subplots(1, ncol, figsize=(3.7 * ncol, 3.9))
if ncol == 1:
    axes = [axes]
for ax, (name, res) in zip(axes, results.items()):
    disp = np.log1p(res["visitation"].copy())   # log scale
    disp[env.walls] = np.nan                     # walls blank
    im = ax.imshow(disp, cmap="viridis")
    ax.scatter([env.start[1]], [env.start[0]], c="lime", s=110, marker="s",
               edgecolors="black", zorder=3)
    ax.scatter([env.goal[1]], [env.goal[0]], c="red", s=150, marker="*",
               edgecolors="black", zorder=3)
    ax.set_title(f"{name}\nstate visitation (log)")
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()


## 12. Plot (c): early intrinsic-reward map

A look *inside* curiosity. Early in training, the intrinsic reward (raw novelty
signal, averaged per cell over the first chunk of episodes) is **high on states
the agent has barely seen** --- exactly the frontier it's pulled toward. As those
states get visited the bonus decays, so the frontier keeps moving outward. The
count-based map ($1/\sqrt{N}$) is smooth; the RND map (neural prediction error) is
the noisy, learned approximation of the same thing. (Cells never visited early are
blank.)


In [ ]:
intr_methods = [m for m in ["count", "RND", "ICM"] if m in results]
fig, axes = plt.subplots(1, len(intr_methods), figsize=(4.2 * len(intr_methods), 3.9))
if len(intr_methods) == 1:
    axes = [axes]
env = GridWorld(CFG)
for ax, name in zip(axes, intr_methods):
    grid = results[name]["intrinsic_grid"].copy()
    grid[grid == 0.0] = np.nan          # unvisited-early cells blank
    grid[env.walls] = np.nan
    im = ax.imshow(grid, cmap="magma")
    ax.scatter([env.start[1]], [env.start[0]], c="lime", s=110, marker="s",
               edgecolors="black", zorder=3)
    ax.scatter([env.goal[1]], [env.goal[0]], c="cyan", s=150, marker="*",
               edgecolors="black", zorder=3)
    ax.set_title(f"{name}: mean intrinsic reward\n(early training)")
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()


## 13. Takeaways

* **Sparse reward turns exploration into the whole problem.** When reward is $0$
  until a distant goal, the value landscape is flat and $\varepsilon$-greedy is
  just an undirected random walk. Through four-rooms doorways under a tight step
  cap, it almost never reaches the goal within budget --- so it almost never
  learns. The visitation heatmap shows it stuck near the start, and its
  first-goal episode (if any) is very late.

* **Intrinsic motivation manufactures a dense signal.** Adding a curiosity bonus
  $r^{\text{total}} = r^{\text{ext}} + \beta\, r^{\text{int}}$ rewards visiting
  novel states, which *systematically* pushes the agent outward until it reaches
  the real goal --- after which ordinary value learning exploits it. All curiosity
  agents reach the goal **far earlier** than $\varepsilon$-greedy and spread
  visitation across the map.

* **Count-based is the clean reference; RND is the neural generalisation.**
  Tabular $1/\sqrt{N(s)}$ is a *perfect* novelty estimate and converges crisply.
  **RND** has no table: a frozen random target + a trained predictor turn "states
  my predictor hasn't fit yet" into a *learned soft count* --- the only thing you
  can do when $s$ is high-dimensional and you can't enumerate it. The price is a
  **noisier** signal: you saw RND's success curve climb more raggedly, and it
  needed care (a separate non-episodic intrinsic value head + an exploration
  floor) to avoid getting trapped in already-explored interior cells. That
  fragility is itself the lesson about neural curiosity.

* **It's the *exploration signal*, not the learner.** Every agent shares the same
  Q-learning core and budget; only the reward/exploration differs. The same idea
  plugs straight into a DQN: replace the tabular update with the function
  approximator and target network from
  [DQN & Rainbow](https://app.notion.com/p/37f95008d76681f396e7ffa88cb535e7) ---
  in fact RND debuted as an exploration bonus *for* deep value/policy agents.

* **Where this sits in the taxonomy.** Intrinsic motivation is orthogonal to the
  model-free vs model-based / value vs policy axes of
  [Part 2: Kinds of RL Algorithms](https://app.notion.com/p/37895008d76681a8861fc7a364b26b68):
  it's an **exploration augmentation** you bolt onto almost any algorithm by
  reshaping the reward. (ICM, by learning *controllable* features via an inverse
  model, additionally targets robustness to uncontrollable noise.)

* **Caveats.** This is a deliberately tiny, deterministic toy with one-hot states.
  Numbers will jitter across seeds (curiosity exploration is stochastic), and the
  neural agents are sensitive to `rnd_scale` / `gamma_int` / the exploration
  floor. Real hard-exploration domains add stochastic dynamics, high-dim
  observations, and the noisy-TV trap (where naive prediction-error curiosity gets
  stuck on noise --- RND's randomness-not-stochasticity design and ICM's
  inverse-feature trick are partial answers). Treat these results as an
  illustration of the *mechanism*, not a benchmark.
